<img src="https://docs.dask.org/en/latest/_images/dask_horizontal.svg"
     align="right"
     width="30%"
     alt="Dask logo">

# Quick demo: `dask.delayed` vs `futures`

- `dask.delayed`: **build a task graph**, then run it with `.compute()`
- `futures`: **submit work immediately** to a scheduler and get `Future` handles back


In [1]:
import time
import dask
from dask.distributed import Client

# Local, in-process cluster (good for demos)
client = Client(n_workers=2, threads_per_worker=1)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:49195,Workers: 2
Dashboard: http://127.0.0.1:8787/status,Total threads: 2
Started: Just now,Total memory: 16.00 GiB
Comm: tcp://127.0.0.1:49202,Total threads: 1
Dashboard: http://127.0.0.1:49203/status,Memory: 8.00 GiB
Nanny: tcp://127.0.0.1:49198,


## 1) `dask.delayed` (lazy)

With `delayed`, calling the function **does not run it**. It returns a lazy object (a node in a task graph). Computation happens when you call `.compute()`.

In [ ]:
@dask.delayed
def inc(x):
    time.sleep(0.2)
    return x + 1

@dask.delayed
def add(x, y):
    time.sleep(0.2)
    return x + y

x = inc(1)
y = inc(2)
z = add(x, y)

print(z)  # lazy (not computed yet)
z.compute()

Delayed('add-3c2ce9fe-7141-4317-94f6-aed203b34e36')


5

## 2) `futures` (eager)

With futures, `client.submit(...)` **starts work immediately** (as resources are available). You get back a `Future` you can query, pass into other tasks, and eventually call `.result()` on.

In [ ]:
# Demonstrating futures execution with Dask distributed

import time
# client is already defined as the Dask distributed client above and is connected.

def inc_now(x):
    time.sleep(0.2)
    return x + 1

def add_now(x, y):
    time.sleep(0.2)
    return x + y

# Submit tasks to the cluster immediately as "futures"
future_x = client.submit(inc_now, 1)

# Wait for computation to finish and retrieve the result
result = future_x.result()
result

2

In [5]:
client.close()

In [ ]:
from dask.distributed import Client, wait
import time